In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Self-contained cell: imports, graph loading, removals, SICP runs, plotting
import matplotlib.pyplot as plt

from main import load_graph_and_communities, run_configured_sicp_intermediates
from removal import (
    ept_out_strength,
    ept_total_strength,
    random_based_removal,
    hyperdegree_based_removal,
    degree_based_removal,
    remove_nodes,
)

# Parameters
BETA = 0.02
T = 25
RUNS = 10
rng_seed = 175
p = 0.20
seed_iterations = 150

# Load graph and communities (this cell can run on its own)
temp_graph, communities = load_graph_and_communities("Restaurants-Rev", tau=2, verbose=True)

# Prepare graph and calculate K
all_nodes = set.union(*temp_graph.values())
N_ = len(all_nodes)
K = int(p * float(N_))

print(f"Running removal strategies with p={p}, K={K} nodes to remove")
print(f"Graph has {len(temp_graph)} hyperedges and {N_} nodes")

# Apply removal strategies
graph_degree = remove_nodes(temp_graph, degree_based_removal(temp_graph, K))
graph_hyperdegree = remove_nodes(temp_graph, hyperdegree_based_removal(temp_graph, K))
graph_random = remove_nodes(temp_graph, random_based_removal(temp_graph, K))
graph_ept = remove_nodes(temp_graph, ept_out_strength(temp_graph, BETA, K))
graph_total_ept = remove_nodes(temp_graph, ept_total_strength(temp_graph, BETA, K))

print("Running SICP for each strategy with intermediate timesteps...")

# Run SICP for each strategy and collect intermediate results
prevalence_degree = run_configured_sicp_intermediates(graph_degree, seed_iterations, beta=BETA, T=T, runs=RUNS, rng_seed=rng_seed)
prevalence_hyperdegree = run_configured_sicp_intermediates(graph_hyperdegree, seed_iterations, beta=BETA, T=T, runs=RUNS, rng_seed=rng_seed)
prevalence_random = run_configured_sicp_intermediates(graph_random, seed_iterations, beta=BETA, T=T, runs=RUNS, rng_seed=rng_seed)
prevalence_ept = run_configured_sicp_intermediates(graph_ept, seed_iterations, beta=BETA, T=T, runs=RUNS, rng_seed=rng_seed)
prevalence_total_ept = run_configured_sicp_intermediates(graph_total_ept, seed_iterations, beta=BETA, T=T, runs=RUNS, rng_seed=rng_seed)

# Plot comparison over time
fig, ax = plt.subplots(figsize=(10, 6))

timesteps = list(range(T + 1))
strategies = {
    'Degree': prevalence_degree,
    'Hyperdegree': prevalence_hyperdegree,
    'Random': prevalence_random,
    'EPT Out-Strength': prevalence_ept,
    'EPT Total-Strength': prevalence_total_ept,
}

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', "#7C0055"]
color_map = {strategy: colors[i] for i, strategy in enumerate(strategies.keys())}

for strategy, prevalence in strategies.items():
    ax.plot(
        timesteps,
        prevalence,
        marker='o',
        label=strategy,
        linewidth=2,
        color=color_map[strategy],
        markersize=4,
    )

ax.set_xlabel('Timestep', fontsize=12)
ax.set_ylabel('Mean Infected Nodes', fontsize=12)
ax.set_title(f'SICP Infection Prevalence by Removal Strategy (p={p}, K={K})', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper left', fontsize=11)

plt.tight_layout()
plt.show()

print(f"\nFinal infected counts:")
for strategy, prevalence in strategies.items():
    print(f"  {strategy}: {prevalence[-1]:.1f} nodes")

# --- Additional plot: percent differences vs Degree (zoomed to show small differences) ---
import numpy as np

baseline = np.array(prevalence_degree)
pct_map = {}
for name, prev in strategies.items():
    if name == 'Degree':
        continue
    prev_arr = np.array(prev)
    with np.errstate(divide='ignore', invalid='ignore'):
        pct = np.where(baseline > 0, (prev_arr - baseline) / baseline * 100.0, 0.0)
    pct_map[name] = pct

fig2, ax2 = plt.subplots(figsize=(10, 4))
for name, pct in pct_map.items():
    ax2.plot(timesteps, pct, marker='o', label=name, linewidth=2, color=color_map[name])

ax2.axhline(0, color='k', linewidth=0.8)
max_abs = max((np.max(np.abs(v)) for v in pct_map.values()), default=0.0)
window = min(max_abs * 1.2 if max_abs > 0 else 0.5, 20.0)
if window < 0.5:
    window = 0.5
ax2.set_ylim(-window, window)
ax2.set_xlabel('Timestep')
ax2.set_ylabel('Percent diff vs Degree (%)')
ax2.set_title('Zoom: Percent difference vs Degree (shows small differences)')
ax2.grid(True, alpha=0.3)
ax2.legend(loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Comprehensive multi-dataset, multi-p comparison with file saves
import csv
import time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

from main import load_graph_and_communities, run_configured_sicp_intermediates
from removal import (
    degree_based_removal,
    hyperdegree_based_removal,
    random_based_removal,
    ept_bridge_out_strength_removal,
    ept_bridge_in_strength_removal,
    ept_boundary_strength_removal,
    ept_bridge_out_fraction_removal,
    ept_bridge_broker_removal,
    ept_boundary_ratio_removal,
    ept_broker_strength,
    ept_pagerank_removal,
    ept_total_strength,
    ept_out_strength,
    remove_nodes,
)

# Parameters
DATASETS = ['Algebra', 'Bars-Rev', 'Music-Rev', 'Restaurants-Rev']
P_VALUES = [0.05, 0.10, 0.20]
BETA = 0.02
T = 25
RUNS = 10
rng_seed = 175
seed_iterations = None  # unbounded

# Output directory
out_dir = Path("outputs/comprehensive_sims")
out_dir.mkdir(parents=True, exist_ok=True)

# Strategy wrapper utilities

def _wrap_no_comm(func):
    return lambda graph, communities, K: func(graph, K)


def _wrap_beta(func):
    return lambda graph, communities, K: func(graph, BETA, K)


def _wrap_comm_beta(func):
    return lambda graph, communities, K: func(graph, communities, BETA, K)


# Strategy list with wrappers so all removal functions can be called uniformly
STRATEGIES = [
    ("degree", _wrap_no_comm(degree_based_removal)),
    ("hyperdegree", _wrap_no_comm(hyperdegree_based_removal)),
    ("random", _wrap_no_comm(random_based_removal)),
    ("ept_out_strength", _wrap_beta(ept_out_strength)),
    ("ept_total_strength", _wrap_beta(ept_total_strength)),
    ("ept_pagerank", _wrap_beta(ept_pagerank_removal)),
    ("ept_broker_strength", _wrap_beta(ept_broker_strength)),
    ("ept_bridge_out_strength", _wrap_comm_beta(ept_bridge_out_strength_removal)),
    ("ept_bridge_in_strength", _wrap_comm_beta(ept_bridge_in_strength_removal)),
    ("ept_boundary_strength", _wrap_comm_beta(ept_boundary_strength_removal)),
    ("ept_bridge_out_fraction", _wrap_comm_beta(ept_bridge_out_fraction_removal)),
    ("ept_bridge_broker", _wrap_comm_beta(ept_bridge_broker_removal)),
    ("ept_boundary_ratio", _wrap_comm_beta(ept_boundary_ratio_removal)),
]

# CSV header
csv_header = ['dataset', 'p', 'strategy', 'K', 'remaining_nodes', 'timestep', 'mean_infected']

overall_start = time.time()
total_tasks = len(DATASETS) * len(P_VALUES) * len(STRATEGIES)
completed_tasks = 0

# Process each dataset
for dataset_idx, dataset in enumerate(DATASETS, 1):
    dataset_start = time.time()
    print(f"\n{'='*70}")
    print(f"[{dataset_idx}/{len(DATASETS)}] Dataset: {dataset}")
    print(f"{'='*70}")
    
    # Load graph once per dataset
    print(f"  Loading graph...")
    temp_graph, communities = load_graph_and_communities(dataset, tau=2, verbose=False)
    all_nodes = set.union(*temp_graph.values())
    N = len(all_nodes)
    print(f"  ✓ Loaded: {len(temp_graph)} hyperedges, {N} nodes")
    
    # Per-dataset output file
    dataset_out_file = out_dir / f"{dataset}_comprehensive_results.csv"
    
    # Write CSV header if file doesn't exist
    if not dataset_out_file.exists():
        with open(dataset_out_file, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=csv_header)
            writer.writeheader()
    
    # Process each p value
    for p_idx, p in enumerate(P_VALUES, 1):
        p_start = time.time()
        K = int(p * float(N))
        print(f"\n  [{p_idx}/{len(P_VALUES)}] p={p:.2f} (K={K} nodes to remove)")
        
        # Apply removal strategies
        print(f"    Computing removal strategies...")
        removed_graphs = {}
        strategy_start = time.time()
        for strat_name, strat_func in STRATEGIES:
            try:
                nodes_to_remove = strat_func(temp_graph, communities, K)
                removed_graphs[strat_name] = remove_nodes(temp_graph, nodes_to_remove)
                print(f"      ✓ {strat_name}")
            except Exception as e:
                print(f"      ✗ {strat_name}: {type(e).__name__}")
                removed_graphs[strat_name] = None
        
        strategy_elapsed = time.time() - strategy_start
        print(f"    Removal strategies done in {strategy_elapsed:.1f}s")
        
        # Run SICP for each strategy and save results
        print(f"    Running SICP simulations...")
        rows = []
        sicp_start = time.time()
        for strat_idx, (strat_name, strat_func) in enumerate(STRATEGIES, 1):
            completed_tasks += 1
            if removed_graphs[strat_name] is None:
                print(f"      [{strat_idx}/{len(STRATEGIES)}] {strat_name:<20} skipped (removal failed)")
                continue
            
            g_removed = removed_graphs[strat_name]
            remaining_nodes = len(set.union(*g_removed.values())) if g_removed else 0
            
            # Run SICP with intermediate results
            try:
                mean_counts = run_configured_sicp_intermediates(
                    g_removed,
                    seed_iterations=seed_iterations,
                    beta=BETA,
                    T=T,
                    runs=RUNS,
                    rng_seed=rng_seed,
                )
                
                # Ensure we have exactly T+1 timesteps
                if len(mean_counts) < T + 1:
                    mean_counts = mean_counts + [mean_counts[-1]] * (T + 1 - len(mean_counts))
                elif len(mean_counts) > T + 1:
                    mean_counts = mean_counts[:T+1]
                
                # Save all timesteps to CSV
                for t, mean_inf in enumerate(mean_counts):
                    rows.append({
                        'dataset': dataset,
                        'p': p,
                        'strategy': strat_name,
                        'K': K,
                        'remaining_nodes': remaining_nodes,
                        'timestep': t,
                        'mean_infected': mean_inf,
                    })
                
                final_pct = 100.0 * mean_counts[-1] / remaining_nodes if remaining_nodes > 0 else 0.0
                print(f"      [{strat_idx}/{len(STRATEGIES)}] {strat_name:<20} final={mean_counts[-1]:6.1f} ({final_pct:5.1f}%)")
            except Exception as e:
                print(f"      [{strat_idx}/{len(STRATEGIES)}] {strat_name:<20} FAILED: {type(e).__name__}")
        
        sicp_elapsed = time.time() - sicp_start
        
        # Append rows to CSV
        if rows:
            with open(dataset_out_file, 'a', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=csv_header)
                writer.writerows(rows)
            print(f"    ✓ Saved {len(rows)} results in {sicp_elapsed:.1f}s")
        
        p_elapsed = time.time() - p_start
        print(f"    Total for p={p:.2f}: {p_elapsed:.1f}s")
    
    dataset_elapsed = time.time() - dataset_start
    print(f"  Dataset {dataset} complete in {dataset_elapsed:.1f}s")

overall_elapsed = time.time() - overall_start
print(f"\n{'='*70}")
print(f"✓ All simulations complete!")
print(f"  Total time: {overall_elapsed:.1f}s ({overall_elapsed/60:.1f}m)")
print(f"  Results saved to: {out_dir}")
print(f"{'='*70}")


In [ ]:
# Graph the overnight_sim results and save images to a folder
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Load the saved results from overnight_outputs/presentation_sims
csv_dir = Path("overnight_outputs/comprehensive_sims")
pres_sims = Path("overnight_outputs/presentation_sims2")
output_dir = Path("overnight_outputs/graphs_presentation")
output_dir.mkdir(parents=True, exist_ok=True)

# all_files = sorted([p for p in csv_dir.glob("*_comprehensive_results.csv") if p.stem.split("_")[0] in ["Bars-Rev"]])
all_files = sorted([csv_dir.glob("Music-Rev_comprehensive_results.csv")])
if not all_files:
    raise FileNotFoundError(f"No CSV files found in {csv_dir}")

music_avg_hdeg = [pres_sims.glob("Music-Rev_comprehensive_results.csv")]

print(list(all_files))
print(list(music_avg_hdeg))

results = pd.concat((pd.read_csv(path) for path in all_files), ignore_index=True)
results['p'] = results['p'].astype(float)
results['remaining_nodes'] = results['remaining_nodes'].astype(int)
results['timestep'] = results['timestep'].astype(int)
results['mean_infected'] = results['mean_infected'].astype(float)

# Preserve the strategy order from the data if possible, otherwise sort alphabetically.
strategy_order = [
    'degree', 'hyperdegree', 'random',
    'ept_total_strength'
    'ept_bridge_in_strength', 'avg_hyperedge_size'
]
present_strategies = [s for s in strategy_order if s in results['strategy'].unique()]
remaining = [s for s in results['strategy'].unique() if s not in present_strategies]
strategy_order = present_strategies + sorted(remaining)

# Setup plot style
plt.style.use('seaborn-v0_8-deep')
colors = plt.get_cmap('tab20').colors
strategy_colors = {s: colors[i % len(colors)] for i, s in enumerate(strategy_order)}

# Helper functions

def save_figure(fig, name):
    path = output_dir / name
    fig.savefig(path, dpi=200, bbox_inches='tight')
    plt.close(fig)


def plot_prevalence_by_timestep(dataset, p_value, df):
    fig, ax = plt.subplots(figsize=(10, 6))
    for strategy in strategy_order:
        sub = df[df['strategy'] == strategy]
        if sub.empty:
            continue
        ax.plot(sub['timestep'], sub['mean_infected'], marker='o', label=strategy, color=strategy_colors[strategy], linewidth=2)

    ax.set_title(f"Mean Infected over Time — {dataset} (p={p_value:.2f})")
    ax.set_xlabel('Timestep')
    ax.set_ylabel('Mean Infected Nodes')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper left', fontsize='small', ncol=2)
    save_figure(fig, f"line_{dataset.replace(' ', '_')}_p{int(p_value*100)}.png")


def plot_final_prevalence_bar(dataset, p_value, df):
    final = df[df['timestep'] == df['timestep'].max()].copy()
    means = final.groupby('strategy')['mean_infected'].mean().reindex(strategy_order).dropna()
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(means.index, means.values, color=[strategy_colors[s] for s in means.index])
    ax.set_title(f"Final Mean Infected at T={df['timestep'].max()} — {dataset} (p={p_value:.2f})")
    ax.set_xlabel('Strategy')
    ax.set_ylabel('Mean Infected Nodes')
    ax.set_xticks(range(len(means.index)))
    ax.set_xticklabels(means.index, rotation=45, ha='right')
    ax.grid(axis='y', alpha=0.3)
    save_figure(fig, f"final_bar_{dataset.replace(' ', '_')}_p{int(p_value*100)}.png")


def plot_percent_difference_vs_degree(dataset, p_value, df):
    pivot = df.pivot_table(index='timestep', columns='strategy', values='mean_infected')
    if 'degree' not in pivot.columns:
        return
    baseline = pivot['degree']
    fig, ax = plt.subplots(figsize=(10, 6))
    for strategy in strategy_order:
        if strategy == 'degree' or strategy not in pivot.columns:
            continue
        diff = (pivot[strategy] - baseline) / baseline.replace({0: np.nan}) * 100.0
        ax.plot(diff.index, diff, marker='o', label=strategy, color=strategy_colors[strategy], linewidth=2)

    ax.axhline(0, color='black', linestyle='--', linewidth=1)
    ax.set_title(f"Percent Difference vs Degree Baseline — {dataset} (p={p_value:.2f})")
    ax.set_xlabel('Timestep')
    ax.set_ylabel('Percent difference vs Degree (%)')
    ax.set_ylim(-50, 50)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper left', fontsize='small', ncol=2)
    save_figure(fig, f"pct_diff_degree_{dataset.replace(' ', '_')}_p{int(p_value*100)}.png")


def plot_summary_final_by_strategy(df):
    final = df[df['timestep'] == df['timestep'].max()].copy()
    pivot = final.pivot_table(index=['dataset', 'p'], columns='strategy', values='mean_infected')
    pivot = pivot.reindex(columns=strategy_order)

    x = np.arange(len(pivot.index))
    labels = [f"{ds}, p={p:.2f}" for ds, p in pivot.index]

    fig, ax = plt.subplots(figsize=(14, 7))
    for strategy in strategy_order:
        if strategy not in pivot.columns:
            continue
        ax.plot(x, pivot[strategy].values, marker='o', label=strategy, color=strategy_colors[strategy], linewidth=2)

    ax.set_title('Final Mean Infected by Strategy for Each Dataset and p')
    ax.set_xlabel('Dataset, p')
    ax.set_ylabel('Mean Infected Nodes at Final T')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper left', fontsize='small', ncol=2)
    save_figure(fig, 'summary_final_mean_infected.png')


def plot_relative_final_difference(df):
    final = df[df['timestep'] == df['timestep'].max()].copy()
    pivot = final.pivot_table(index=['dataset', 'p'], columns='strategy', values='mean_infected')
    if 'degree' not in pivot.columns:
        return
    baseline = pivot['degree']
    rel = (pivot.subtract(baseline, axis=0).divide(baseline.replace({0: np.nan}), axis=0) * 100.0)
    rel = rel.reindex(columns=strategy_order)

    x = np.arange(len(rel.index))
    labels = [f"{ds}, p={p:.2f}" for ds, p in rel.index]

    fig, ax = plt.subplots(figsize=(14, 7))
    for strategy in strategy_order:
        if strategy == 'degree' or strategy not in rel.columns:
            continue
        ax.plot(x, rel[strategy].values, marker='o', label=strategy, color=strategy_colors[strategy], linewidth=2)

    ax.axhline(0, color='black', linestyle='--', linewidth=1)
    ax.set_title('Final Percent Difference vs Degree Baseline by Dataset and p')
    ax.set_xlabel('Dataset, p')
    ax.set_ylabel('Percent difference vs Degree at final T (%)')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper left', fontsize='small', ncol=2)
    save_figure(fig, 'summary_final_percent_difference_vs_degree.png')


# Generate graphs for each dataset and p combination
for dataset in sorted(results['dataset'].unique()):
    for p_value in sorted(results[results['dataset'] == dataset]['p'].unique()):
        subset = results[(results['dataset'] == dataset) & (results['p'] == p_value)]
        plot_prevalence_by_timestep(dataset, p_value, subset)
        plot_final_prevalence_bar(dataset, p_value, subset)
        plot_percent_difference_vs_degree(dataset, p_value, subset)

# Generate summary graphs across datasets and p values
plot_summary_final_by_strategy(results)

plot_relative_final_difference(results)
plot_relative_final_difference(results)
print(f"Saved graph images to: {output_dir}")


print(f"Saved graph images to: {output_dir}")

In [ ]:
# Plot intermediate Music-Rev results for p=0.20
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

csv_file = Path("overnight_outputs/comprehensive_sims/Music-Rev_comprehensive_results.csv")
output_dir = Path("overnight_outputs/graphs")
output_dir.mkdir(parents=True, exist_ok=True)

if not csv_file.exists():
    raise FileNotFoundError(f"Expected file not found: {csv_file}")

music_df = pd.read_csv(csv_file)
music_df['p'] = music_df['p'].astype(float)
music_df['timestep'] = music_df['timestep'].astype(int)
music_df['mean_infected'] = music_df['mean_infected'].astype(float)

subset = music_df[music_df['p'] == 0.20].sort_values(['strategy', 'timestep'])
if subset.empty:
    raise ValueError("No Music-Rev rows found for p=0.20")

plt.figure(figsize=(10, 6))
for strategy in subset['strategy'].unique():
    if strategy not in ['degree', 'hyperdegree', 'ept_bridge_in_strength', 'ept_pagerank', 'ept_total_strength']: continue
    sub = subset[subset['strategy'] == strategy]
    plt.plot(sub['timestep'], sub['mean_infected'], marker='o', linewidth=2, label=strategy)

plt.title('Music-Rev: Mean Infected over Time (p=0.20)')
plt.xlabel('Timestep')
plt.ylabel('Mean Infected Nodes')
plt.grid(True, alpha=0.3)
plt.legend(loc='upper left', fontsize='small', ncol=1)
plt.tight_layout()
output_path = output_dir / 'Music-Rev_p20_intermediate.png'
plt.savefig(output_path, dpi=200)
plt.show()
print(f"Saved Music-Rev p=0.20 intermediate graph to: {output_path}")

In [ ]:
# Plot final results bar chart for every dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

csv_dir = Path("overnight_outputs/comprehensive_sims")
output_dir = Path("overnight_outputs/graphs")
output_dir.mkdir(parents=True, exist_ok=True)

all_files = sorted(csv_dir.glob("*_comprehensive_results.csv"))
if not all_files:
    raise FileNotFoundError(f"No CSV files found in {csv_dir}")

df = pd.concat((pd.read_csv(path) for path in all_files), ignore_index=True)
df['p'] = df['p'].astype(float)
df['timestep'] = df['timestep'].astype(int)
df['mean_infected'] = df['mean_infected'].astype(float)

# Use the final timestep for each dataset/p/strategy combination
final_df = df.loc[df.groupby(['dataset', 'p', 'strategy'])['timestep'].idxmax()].reset_index(drop=True)

for dataset in sorted(final_df['dataset'].unique()):
    subset = final_df[final_df['dataset'] == dataset]
    pivot = subset.pivot(index='strategy', columns='p', values='mean_infected')
    if pivot.empty:
        continue

    # Keep degree first for consistent ordering
    order = sorted(pivot.index, key=lambda s: (s != 'degree', s))
    pivot = pivot.reindex(order)

    fig, ax = plt.subplots(figsize=(12, 6))
    pivot.plot(kind='bar', ax=ax)
    ax.set_title(f'Final Mean Infected by Strategy — {dataset}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Strategy')
    ax.set_ylabel('Mean Infected Nodes at Final T')
    ax.grid(axis='y', alpha=0.3)
    ax.legend(title='p', loc='upper right')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()

    output_path = output_dir / f"{dataset.replace(' ', '_')}_final_bar_chart.png"
    fig.savefig(output_path, dpi=200)
    plt.show()
    print(f"Saved final bar chart for {dataset} to: {output_path}")


In [ ]:
# Single experiment: contact-high-school, no nodes removed, immediate intermediate plot
import matplotlib.pyplot as plt

from main import load_graph_and_communities, run_configured_sicp_intermediates

BETA = 0.2
T =100
RUNS = 10
RNG_SEED = 175
SEED_ITERATIONS = None

graph, communities = load_graph_and_communities("contact-high-school", tau=2, verbose=True)

print(f"Loaded contact-high-school graph with {len(graph)} hyperedges")

prevalence = run_configured_sicp_intermediates(
    graph,
    seed_iterations=SEED_ITERATIONS,
    beta=BETA,
    T=T,
    runs=RUNS,
    rng_seed=RNG_SEED,
)

timesteps = list(range(len(prevalence)))

plt.figure(figsize=(10, 6))
plt.plot(timesteps, prevalence, marker='o', linewidth=2)
plt.title('Contact-High-School: Mean Infected over Time (0 nodes removed)')
plt.xlabel('Timestep')
plt.ylabel('Mean Infected Nodes')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final mean infected at T={T}: {prevalence[-1]:.2f}")

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# --- Directories ---
csv_dir = Path("overnight_outputs/comprehensive_sims")
pres_sims_dir = Path("overnight_outputs/presentation_sims2")
output_dir = Path("overnight_outputs/graphs_presentation/music_rev")
output_dir.mkdir(parents=True, exist_ok=True)

DATASET_NAME = "Music-Rev"

# --- Desired strategies ---
# from comprehensive_sims:
core_strategies = {
    "degree",
    "hyperdegree",
    "random",
    "ept_total_strength",
    "ept_bridge_in_strength",     # common alt name in many outputs
}

# from presentation_sims2:
avg_strategy_aliases = {
    "avg_hyperedge_size",         # appears in your original strategy_order
}

# --- Utility: load CSV(s) robustly ---
def read_many_csv(paths):
    frames = []
    for p in paths:
        try:
            df = pd.read_csv(p)
            df["__source_file__"] = p.name
            frames.append(df)
        except Exception as e:
            print(f"[WARN] Failed reading {p}: {e}")
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

def save_figure(fig, name):
    path = output_dir / name
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)

# --- Find relevant CSVs ---
# comprehensive results for Music-Rev
comp_files = sorted(csv_dir.glob(f"{DATASET_NAME}*_comprehensive_results.csv"))
if not comp_files:
    # fallback: maybe filename format differs slightly
    comp_files = sorted([p for p in csv_dir.glob("*.csv") if DATASET_NAME in p.name and "comprehensive" in p.name])

if not comp_files:
    raise FileNotFoundError(f"No comprehensive CSVs found for {DATASET_NAME} in {csv_dir}")

# presentation sims files (to extract avg_hyperdegree_size strategy)
pres_files = sorted([p for p in pres_sims_dir.glob("*.csv") if DATASET_NAME in p.name or "Music" in p.name])
# If your presentation csvs don't include the dataset name in filename, remove filtering:
# pres_files = sorted(pres_sims_dir.glob("*.csv"))

# --- Load data ---
comp = read_many_csv(comp_files)
pres = read_many_csv(pres_files) if pres_files else pd.DataFrame()

# --- Basic validation / required columns ---
required_cols = {"dataset", "p", "strategy", "timestep", "mean_infected"}
missing = required_cols - set(comp.columns)
if missing:
    raise ValueError(f"Comprehensive CSVs missing required columns: {missing}")

# Ensure presentation has required cols if not empty
if not pres.empty:
    pres_missing = required_cols - set(pres.columns)
    if pres_missing:
        raise ValueError(f"Presentation CSVs missing required columns: {pres_missing}")

# --- Filter to Music-Rev only ---
comp = comp[comp["dataset"] == DATASET_NAME].copy()
if not pres.empty:
    # Some presentation sims might not fill dataset column consistently.
    # If dataset exists but differs, we keep rows where dataset matches OR file name indicates it's Music-Rev.
    pres = pres[(pres["dataset"] == DATASET_NAME) | (pres["__source_file__"].str.contains("Music", case=False, na=False))].copy()

# --- Normalize types ---
def coerce_types(df):
    df["p"] = df["p"].astype(float)
    df["timestep"] = df["timestep"].astype(int)
    # remaining_nodes might not exist in pres; handle gracefully
    if "remaining_nodes" in df.columns:
        df["remaining_nodes"] = df["remaining_nodes"].astype(int)
    df["mean_infected"] = df["mean_infected"].astype(float)
    return df

comp = coerce_types(comp)
if not pres.empty:
    pres = coerce_types(pres)

# --- Normalize strategy names ---
# Canonicalize ept_bridge_in_strength -> ept_bridge_in
strategy_rename = {
    "ept_bridge_in_strength": "ept_bridge_in",
    "avg_hyperedge_size": "avg_hyperdegree_size",  # unify on the name you requested
}
comp["strategy"] = comp["strategy"].replace(strategy_rename)
if not pres.empty:
    pres["strategy"] = pres["strategy"].replace(strategy_rename)

# --- Keep only desired strategies ---
comp = comp[comp["strategy"].isin({"degree","hyperdegree","random","ept_total_strength","ept_bridge_in"})].copy()

# pull only avg_hyperdegree_size from presentation_sims2
avg_df = pd.DataFrame()
if not pres.empty:
    avg_df = pres[pres["strategy"].isin({"avg_hyperdegree_size"})].copy()

# If avg strategy wasn't already canonical in pres, try aliases:
if avg_df.empty and not pres.empty:
    avg_df = pres[pres["strategy"].isin(avg_strategy_aliases)].copy()
    avg_df["strategy"] = "avg_hyperdegree_size"

# --- Combine ---
results = pd.concat([comp, avg_df], ignore_index=True)

# --- Define final strategy order (exactly what you want in plots) ---
strategy_order = [
    "degree",
    "hyperdegree",
    "random",
    "ept_total_strength",
    "ept_bridge_in",
    "avg_hyperdegree_size",
]

# keep only those present
strategy_order = [s for s in strategy_order if s in results["strategy"].unique()]

# --- Plot style ---
plt.style.use("seaborn-v0_8-deep")
colors = plt.get_cmap("tab20").colors
print(colors)
strategy_colors = {s: colors[i % len(colors)] for i, s in enumerate(strategy_order)}
print(strategy_colors)
raise
# --- Plot functions ---
def plot_prevalence_by_timestep(dataset, p_value, df):
    fig, ax = plt.subplots(figsize=(10, 6))
    for strategy in strategy_order:
        sub = df[df["strategy"] == strategy]
        if sub.empty:
            continue
        ax.plot(
            sub["timestep"], sub["mean_infected"],
            marker="o", label=strategy,
            color=strategy_colors[strategy], linewidth=2
        )
    ax.set_title(f"Mean Infected over Time — {dataset} (p={p_value:.2f})")
    ax.set_xlabel("Timestep")
    ax.set_ylabel("Mean Infected Nodes")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper left", fontsize="small", ncol=2)
    save_figure(fig, f"line_{dataset.replace(' ', '_')}_p{int(p_value*100)}.png")

def plot_final_prevalence_bar(dataset, p_value, df):
    final_t = df["timestep"].max()
    final = df[df["timestep"] == final_t].copy()
    means = final.groupby("strategy")["mean_infected"].mean().reindex(strategy_order).dropna()

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(means.index, means.values, color=[strategy_colors[s] for s in means.index])
    ax.set_title(f"Final Mean Infected at T={final_t} — {dataset} (p={p_value:.2f})")
    ax.set_xlabel("Strategy")
    ax.set_ylabel("Mean Infected Nodes")
    ax.set_xticks(range(len(means.index)))
    ax.set_xticklabels(means.index, rotation=45, ha="right")
    ax.grid(axis="y", alpha=0.3)
    save_figure(fig, f"final_bar_{dataset.replace(' ', '_')}_p{int(p_value*100)}.png")

def plot_percent_difference_vs_degree(dataset, p_value, df):
    pivot = df.pivot_table(index="timestep", columns="strategy", values="mean_infected", aggfunc="mean")
    if "degree" not in pivot.columns:
        return

    baseline = pivot["degree"]
    fig, ax = plt.subplots(figsize=(10, 6))

    for strategy in strategy_order:
        if strategy == "degree" or strategy not in pivot.columns:
            continue
        diff = (pivot[strategy] - baseline) / baseline.replace({0: np.nan}) * 100.0
        ax.plot(diff.index, diff, marker="o", label=strategy, color=strategy_colors[strategy], linewidth=2)

    ax.axhline(0, color="black", linestyle="--", linewidth=1)
    ax.set_title(f"Percent Difference vs Degree Baseline — {dataset} (p={p_value:.2f})")
    ax.set_xlabel("Timestep")
    ax.set_ylabel("Percent difference vs Degree (%)")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper left", fontsize="small", ncol=2)
    save_figure(fig, f"pct_diff_degree_{dataset.replace(' ', '_')}_p{int(p_value*100)}.png")

def plot_summary_final_by_strategy(df):
    final_t = df["timestep"].max()
    final = df[df["timestep"] == final_t].copy()

    pivot = final.pivot_table(index=["dataset", "p"], columns="strategy", values="mean_infected", aggfunc="mean")
    pivot = pivot.reindex(columns=strategy_order)

    x = np.arange(len(pivot.index))
    labels = [f"{ds}, p={p:.2f}" for ds, p in pivot.index]

    fig, ax = plt.subplots(figsize=(14, 7))
    for strategy in strategy_order:
        if strategy not in pivot.columns:
            continue
        ax.plot(x, pivot[strategy].values, marker="o", label=strategy,
                color=strategy_colors[strategy], linewidth=2)

    ax.set_title(f"Final Mean Infected by Strategy — {DATASET_NAME} (T={final_t})")
    ax.set_xlabel("Dataset, p")
    ax.set_ylabel("Mean Infected Nodes at Final T")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper left", fontsize="small", ncol=2)
    save_figure(fig, "summary_final_mean_infected.png")

def plot_relative_final_difference(df):
    final_t = df["timestep"].max()
    final = df[df["timestep"] == final_t].copy()

    pivot = final.pivot_table(index=["dataset", "p"], columns="strategy", values="mean_infected", aggfunc="mean")
    if "degree" not in pivot.columns:
        return

    baseline = pivot["degree"]
    rel = pivot.subtract(baseline, axis=0).divide(baseline.replace({0: np.nan}), axis=0) * 100.0
    rel = rel.reindex(columns=strategy_order)

    x = np.arange(len(rel.index))
    labels = [f"{ds}, p={p:.2f}" for ds, p in rel.index]

    fig, ax = plt.subplots(figsize=(14, 7))
    for strategy in strategy_order:
        if strategy == "degree" or strategy not in rel.columns:
            continue
        ax.plot(x, rel[strategy].values, marker="o", label=strategy,
                color=strategy_colors[strategy], linewidth=2)

    ax.axhline(0, color="black", linestyle="--", linewidth=1)
    ax.set_title(f"Final Percent Difference vs Degree — {DATASET_NAME} (T={final_t})")
    ax.set_xlabel("Dataset, p")
    ax.set_ylabel("Percent difference vs Degree at final T (%)")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper left", fontsize="small", ncol=2)
    save_figure(fig, "summary_final_percent_difference_vs_degree.png")

# --- Generate graphs (Music-Rev only) ---
results = results[results["dataset"] == DATASET_NAME].copy()

for p_value in sorted(results["p"].unique()):
    subset = results[results["p"] == p_value].copy()
    plot_prevalence_by_timestep(DATASET_NAME, p_value, subset)
    plot_final_prevalence_bar(DATASET_NAME, p_value, subset)
    plot_percent_difference_vs_degree(DATASET_NAME, p_value, subset)

plot_summary_final_by_strategy(results)
plot_relative_final_difference(results)

print(f"Saved graph images to: {output_dir.resolve()}")
print(f"Loaded comprehensive files: {[p.name for p in comp_files]}")
print(f"Loaded presentation files:  {[p.name for p in pres_files]}")
print(f"Strategies plotted: {strategy_order}")


{'degree': (0.12156862745098039, 0.4666666666666667, 0.7058823529411765), 'hyperdegree': (0.6823529411764706, 0.7803921568627451, 0.9098039215686274), 'random': (1.0, 0.4980392156862745, 0.054901960784313725), 'ept_total_strength': (1.0, 0.7333333333333333, 0.47058823529411764), 'ept_bridge_in': (0.17254901960784313, 0.6274509803921569, 0.17254901960784313), 'avg_hyperdegree_size': (0.596078431372549, 0.8745098039215686, 0.5411764705882353)}


RuntimeError: No active exception to reraise

In [ ]:

strategy_colors = {s: colors[i % len(colors)] for i, s in enumerate(strategy_order)}
